## **Vector representation**

### Environment Setup & Dependencies

This section initializes the computational environment for the vectorization phase:

* **PyTorch & SentenceTransformers**: Essential libraries for loading and executing the pre-trained BERT model (`all-MiniLM-L6-v2`) with GPU acceleration support.
* **Scikit-Learn**: Provides the `TfidfVectorizer` for generating statistical text representations.
* **Joblib & NumPy**: Efficient serialization tools utilized to persist large sparse matrices and dense embedding arrays to disk.

In [2]:
!pip install torch

import torch
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from google.colab import drive

### Vector Space Representation

In this phase, raw textual data is transformed into numerical vector representations suitable for machine learning algorithms. We employ a hybrid feature extraction strategy, generating three distinct vector spaces to address specific analytical requirements:

#### 1. Standard TF-IDF
* **Input Data**: `text_clean` (Normalized text).
* **Methodology**: Term Frequency-Inverse Document Frequency.
* **Configuration**: The vocabulary is limited to the top **5,000** features, including both **unigrams and bigrams**.
* **Purpose**: This matrix serves as the input for supervised baseline models (e.g., Logistic Regression, SVM). It emphasizes keyword frequency and discriminative power while maintaining low computational complexity.

#### 2. BERT Contextual Embeddings
* **Input Data**: `text_review` (Raw text with punctuation).
* **Methodology**: **Sentence-BERT (SBERT)** using the `all-MiniLM-L6-v2` architecture.
* **Configuration**: Generates dense **384-dimensional** vectors.
* **Purpose**: Unlike statistical methods, BERT captures **semantic context** and sentence structure. These embeddings are critical for the **BERTopic** algorithm and for advanced semantic similarity tasks, as they can distinguish nuanced meanings (e.g., "not good" vs. "good").

#### 3. Topic-Specific TF-IDF (For LSA/NMF)
* **Input Data**: `topic_text_clean` (Text with pre-joined bigrams, e.g., `credit_card`).
* **Methodology**: High-dimensional TF-IDF.
* **Configuration**: The vocabulary is expanded to **10,000** features to capture niche themes. `ngram_range` is set to `(1, 1)` because bigrams are already tokenized as single units in the preprocessing phase.
* **Purpose**: Specifically optimized for probabilistic topic modeling algorithms (**LSA**, **NMF**). The larger feature space allows for the detection of granular sub-topics that might be excluded from the classification matrix.

In [3]:
# ==============================================================================
# 1. SETUP AND DATA LOADING
# ==============================================================================
print("[INFO] Phase 1: Environment Setup and Data Loading...")

# Mount Google Drive
if not os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

# Path Configuration
DRIVE_PATH = '/content/drive/MyDrive/MAGISTRALE/Text_Mining/Datasets'
VECTORS_PATH = os.path.join(DRIVE_PATH, 'vectors')
os.makedirs(VECTORS_PATH, exist_ok=True)

# Load Train/Test splits
try:
    print(f"[INFO] Loading datasets from {DRIVE_PATH}...")
    df_train = pd.read_csv(os.path.join(DRIVE_PATH, 'train_dataset.csv'))
    df_test = pd.read_csv(os.path.join(DRIVE_PATH, 'test_dataset.csv'))
except FileNotFoundError:
    print("[ERROR] Train/Test files not found. Please verify the path.")
    exit()

# Data Integrity: Handling missing values and ensuring string types
# text_clean       -> Used for Standard Classification (TF-IDF)
# text_review      -> Used for BERT (requires full context/punctuation)
# topic_text_clean -> Used for Topic Modeling (LSA/NMF)
cols_to_fix = ['text_clean', 'text_review', 'topic_text_clean']

for col in cols_to_fix:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna('').astype(str)
        df_test[col] = df_test[col].fillna('').astype(str)
    else:
        print(f"[WARNING] Column '{col}' missing in dataset.")

print(f"[INFO] Data loaded. Train Samples: {len(df_train)}, Test Samples: {len(df_test)}")

# ==============================================================================
# 2. STANDARD TF-IDF VECTORIZATION (FOR CLASSIFICATION)
# ==============================================================================
print("\n[INFO] Phase 2: Generating TF-IDF Matrix for Classification...")

# Configuration: Optimized for Supervised Learning (SVM, LogReg)
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # Dynamic bigram generation
    min_df=10,
    max_df=0.9
)

print("   -> Fitting vectorizer on Training Set...")
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['text_clean'])
X_test_tfidf = tfidf_vectorizer.transform(df_test['text_clean'])

# Serialization
joblib.dump(X_train_tfidf, os.path.join(VECTORS_PATH, 'tfidf_train.pkl'))
joblib.dump(X_test_tfidf, os.path.join(VECTORS_PATH, 'tfidf_test.pkl'))
joblib.dump(tfidf_vectorizer, os.path.join(VECTORS_PATH, 'tfidf_vectorizer_model.pkl'))
print("[SUCCESS] Classification Vectors saved.")

# ==============================================================================
# 3. BERT EMBEDDINGS GENERATION
# ==============================================================================
print("\n[INFO] Phase 3: Generating Contextual BERT Embeddings...")

# Load pre-trained SBERT model
model_name = 'all-MiniLM-L6-v2'
print(f"   -> Loading model: {model_name}...")
bert_model = SentenceTransformer(model_name)

# Device check (GPU acceleration)
if torch.cuda.is_available():
    print("   -> GPU detected. Inference enabled on CUDA.")
    bert_model = bert_model.to('cuda')
else:
    print("   -> GPU not detected. Running on CPU (slower).")

print("   -> Encoding Training Set...")
X_train_bert = bert_model.encode(
    df_train['text_review'].values,
    show_progress_bar=True,
    batch_size=64
)

print("   -> Encoding Test Set...")
X_test_bert = bert_model.encode(
    df_test['text_review'].values,
    show_progress_bar=True,
    batch_size=64
)

# Serialization (NumPy format for efficiency)
np.save(os.path.join(VECTORS_PATH, 'bert_train.npy'), X_train_bert)
np.save(os.path.join(VECTORS_PATH, 'bert_test.npy'), X_test_bert)
print("[SUCCESS] BERT Embeddings saved.")

# ==============================================================================
# 4. TOPIC-SPECIFIC TF-IDF VECTORIZATION
# ==============================================================================
print("\n[INFO] Phase 4: Generating TF-IDF Matrix for Topic Modeling...")

if 'topic_text_clean' in df_train.columns:
    # Configuration: Optimized for LSA/NMF
    # Note: 'topic_text_clean' already contains joined bigrams (e.g., credit_card).
    # Therefore, we use ngram_range=(1, 1) to treat them as single tokens.
    topic_vectorizer = TfidfVectorizer(
        max_features=10000,   # Expanded vocabulary for topic nuance
        min_df=15,
        max_df=0.90,
        ngram_range=(1, 1)
    )

    print("   -> Fitting Topic Vectorizer on Training Set...")
    X_train_topic = topic_vectorizer.fit_transform(df_train['topic_text_clean'])
    X_test_topic = topic_vectorizer.transform(df_test['topic_text_clean'])

    vocab_size = len(topic_vectorizer.get_feature_names_out())
    print(f"   -> Vocabulary Size: {vocab_size} tokens")

    # Serialization with distinct filenames
    joblib.dump(X_train_topic, os.path.join(VECTORS_PATH, 'tfidf_topic_train.pkl'))
    joblib.dump(X_test_topic, os.path.join(VECTORS_PATH, 'tfidf_topic_test.pkl'))
    joblib.dump(topic_vectorizer, os.path.join(VECTORS_PATH, 'tfidf_topic_vectorizer.pkl'))
    print("[SUCCESS] Topic Modeling Vectors saved.")

else:
    print("[ERROR] Column 'topic_text_clean' not found. Topic Modeling preparation skipped.")

print("\n" + "="*50)
print("[COMPLETED] All Vectorization Tasks Finished.")
print("="*50)

[INFO] Phase 1: Environment Setup and Data Loading...
Mounted at /content/drive
[INFO] Loading datasets from /content/drive/MyDrive/MAGISTRALE/Text_Mining/Datasets...
[INFO] Data loaded. Train Samples: 958686, Test Samples: 239672

[INFO] Phase 2: Generating TF-IDF Matrix for Classification...
   -> Fitting vectorizer on Training Set...
[SUCCESS] Classification Vectors saved.

[INFO] Phase 3: Generating Contextual BERT Embeddings...
   -> Loading model: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

   -> GPU detected. Inference enabled on CUDA.
   -> Encoding Training Set...


Batches:   0%|          | 0/14980 [00:00<?, ?it/s]

   -> Encoding Test Set...


Batches:   0%|          | 0/3745 [00:00<?, ?it/s]

[SUCCESS] BERT Embeddings saved.

[INFO] Phase 4: Generating TF-IDF Matrix for Topic Modeling...
   -> Fitting Topic Vectorizer on Training Set...
   -> Vocabulary Size: 10000 tokens
[SUCCESS] Topic Modeling Vectors saved.

[COMPLETED] All Vectorization Tasks Finished.
